# Phase 8 — BERT-Based Transformer Classification
**Project**: Sarcasm & Irony Detection System using NLP  
**Phase**: 8 — Contextual Transformer Representations (bert-base-uncased)  
**Dataset**: Phase 1 Approved Clean Datasets (`train_clean.csv`: 81,359 / `test_clean.csv`: 5,527)  
**Classes**: Regular (0), Sarcasm (1), Irony (2), Figurative (3)  
**Framework**: Hugging Face Transformers + PyTorch  
**Primary Research Question**: *Can contextual transformer-based representations from BERT improve four-class sarcasm and irony classification compared with traditional NLP (BoW, TF-IDF) and recurrent neural-network approaches (GloVe + Bi-LSTM, FastText + Bi-LSTM + Attention)?*

## 1. Imports

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import transformers

# Project root configuration
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.transformers.bert.tokenizer import BertTokenizerManager
from src.transformers.bert.dataset import load_and_split_data, create_dataloaders, load_label_mapping
from src.transformers.bert.model import BertSarcasmClassifier
from src.transformers.bert.train import train_bert_model
from src.transformers.bert.evaluate import (
    calculate_classification_metrics,
    evaluate_bert_dataloader,
    generate_predictions_dataframe,
    analyze_misclassifications,
    CLASS_NAMES,
)
from src.transformers.bert.predict import BertPredictor
from src.transformers.bert.utils import set_seed, get_environment_info

set_seed(42)
print('Loaded imports and initialized random seed to 42.')

## 2. Environment Check

In [ ]:
env = get_environment_info(seed=42)
print(f"PyTorch Version:      {env['pytorch_version']}")
print(f"Transformers Version: {env['transformers_version']}")
print(f"CUDA Available:       {env['cuda_available']}")
print(f"Active Device:        {env['device_name']}")

## 3. Load Dataset

In [ ]:
train_csv = BASE_DIR / 'data' / 'processed' / 'train_clean.csv'
test_csv = BASE_DIR / 'data' / 'processed' / 'test_clean.csv'
label_map = load_label_mapping(BASE_DIR / 'data' / 'processed' / 'label_mapping.json')

train_df_raw = pd.read_csv(train_csv)
test_df_raw = pd.read_csv(test_csv)

print(f"Full Training Shape: {train_df_raw.shape}")
print(f"Official Test Shape: {test_df_raw.shape}")
print(f"Label Mapping:       {label_map}")
train_df_raw.head()

## 4. Class Distribution

In [ ]:
plt.figure(figsize=(8, 4))
counts = train_df_raw['label'].value_counts()
sns.barplot(x=counts.index, y=counts.values, palette='Blues_r')
plt.title('Training Dataset Class Distribution (Phase 1 Approved)', fontsize=12, fontweight='bold')
plt.ylabel('Sample Count')
plt.xlabel('Category')
for idx, val in enumerate(counts.values):
    plt.text(idx, val + 500, f"{val:,} ({val/len(train_df_raw)*100:.1f}%)", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Train / Validation / Test Split (Strict Leakage Prevention)

In [ ]:
split_data = load_and_split_data(
    train_csv_path=train_csv,
    test_csv_path=test_csv,
    val_size=0.2,
    random_state=42,
)
train_df = split_data['train_df']
val_df = split_data['val_df']
test_df = split_data['test_df']

print(f"Train Split:      {len(train_df):,} samples (80%)")
print(f"Validation Split: {len(val_df):,} samples (20%)")
print(f"Test Set:         {len(test_df):,} samples (Held out completely)")

## 6. BERT Tokenizer

In [ ]:
tokenizer_mgr = BertTokenizerManager(model_name_or_path='bert-base-uncased', max_length=80)
print(f"Vocabulary Size: {tokenizer_mgr.vocab_size}")
print(f"Special Tokens:  {tokenizer_mgr.special_tokens_map}")

# Inspect WordPiece subword tokenization
sample = "#sarcasm is definitely soooo subtle!"
tokens = tokenizer_mgr.inspect_subword_tokens(sample)
print("\nSubword tokenization demonstration:")
for t, tid in tokens:
    print(f"  {t:15s} -> ID {tid}")

## 7. Token-Length Analysis & Sequence Length Justification

In [ ]:
stats = tokenizer_mgr.analyze_sequence_lengths(
    train_df['text'].tolist()[:10000],
    candidate_lengths=[32, 48, 64, 80, 96, 128]
)
print(f"Mean Token Length:   {stats['mean_length']:.2f}")
print(f"Median Token Length: {stats['median_length']:.1f}")
print(f"95th Percentile:     {stats['p95']:.1f}")
print(f"99th Percentile:     {stats['p99']:.1f}")
print(f"Maximum Length:      {stats['max_length']}")

trunc_df = pd.DataFrame(stats['truncation_rates']).T
display(trunc_df)

## 8. Dataset Preparation & PyTorch DataLoaders

In [ ]:
batch_size = 16
max_length = 80

loaders = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    tokenizer_manager=tokenizer_mgr,
    batch_size=batch_size,
    max_length=max_length,
    test_df=test_df,
)

batch = next(iter(loaders['train']))
print(f"Batch input_ids shape:      {batch['input_ids'].shape}")
print(f"Batch attention_mask shape: {batch['attention_mask'].shape}")
print(f"Batch labels shape:         {batch['labels'].shape}")

## 9. BERT Sequence Classification Model

In [ ]:
model = BertSarcasmClassifier(model_name_or_path='bert-base-uncased', num_labels=4)
params = model.count_parameters()
print(f"Total Parameters:         {params['total_parameters']:,}")
print(f"Trainable Parameters:     {params['trainable_parameters']:,}")
print(f"Non-Trainable Parameters: {params['non_trainable_parameters']:,}")

## 10. Training Configuration

In [ ]:
epochs = 3
learning_rate = 2e-5
weight_decay = 0.01
warmup_ratio = 0.1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Configured fine-tuning on {device} with LR={learning_rate}, weight_decay={weight_decay}")

## 11. Model Fine-Tuning Execution

In [ ]:
# Load trained model artifacts and history
res_dir = BASE_DIR / 'results' / 'bert'
history_path = res_dir / 'training_history.csv'

if history_path.exists():
    history_df = pd.read_csv(history_path)
    display(history_df)
else:
    print("Run python -m src.transformers.bert.run_experiments to generate training history.")

## 12. Validation Results

In [ ]:
model_results_path = res_dir / 'model_results.csv'
if model_results_path.exists():
    res_df = pd.read_csv(model_results_path)
    display(res_df)
else:
    print("Model results file pending execution.")

## 13. Training Curves

In [ ]:
curve_img = res_dir / 'plots' / 'training_curves.png'
if curve_img.exists():
    from IPython.display import Image
    display(Image(filename=str(curve_img)))
else:
    print("Training curve image not yet generated.")

## 14. Test Evaluation (Official Untouched Dataset)

In [ ]:
report_path = res_dir / 'classification_report.csv'
if report_path.exists():
    rep_df = pd.read_csv(report_path)
    display(rep_df)
else:
    print("Classification report file pending.")

## 15. Classification Report

In [ ]:
if report_path.exists():
    print(pd.read_csv(report_path).to_string(index=False))

## 16. Confusion Matrix

In [ ]:
cm_img = res_dir / 'plots' / 'confusion_matrix.png'
if cm_img.exists():
    from IPython.display import Image
    display(Image(filename=str(cm_img)))
else:
    cm_path = res_dir / 'confusion_matrix.csv'
    if cm_path.exists():
        display(pd.read_csv(cm_path, index_col=0))

## 17. Error Analysis (Sarcasm ↔ Irony & Figurative Boundaries)

In [ ]:
err_path = res_dir / 'error_analysis.json'
if err_path.exists():
    with open(err_path, 'r') as f:
        err_data = json.load(f)
    print(f"Total Errors: {err_data['total_errors']} / {err_data['total_samples']} ({err_data['error_rate_pct']}%)")
    print(f"Sarcasm ↔ Irony Mutual Confusion: {err_data['sarcasm_irony_mutual_confusion']}")
    print(f"Sarcasm -> Irony: {err_data['sarcasm_as_irony']}")
    print(f"Irony -> Sarcasm: {err_data['irony_as_sarcasm']}")
    print("\nTop Confident Errors:")
    display(pd.DataFrame(err_data['top_high_confidence_errors']))

## 18. Cross-Phase Model Comparison

In [ ]:
prog_path = BASE_DIR / 'results' / 'model_progression.csv'
if prog_path.exists():
    prog_df = pd.read_csv(prog_path)
    display(prog_df.sort_values(by='macro_f1', ascending=False))

## 19. Computational Comparison

In [ ]:
comp_path = BASE_DIR / 'results' / 'model_computational_comparison.csv'
if comp_path.exists():
    comp_df = pd.read_csv(comp_path)
    display(comp_df)

## 20. Live Inference Test

In [ ]:
saved_model_dir = BASE_DIR / 'models' / 'transformers' / 'bert'
if (saved_model_dir / 'model').exists():
    predictor = BertPredictor.load_from_directory(saved_model_dir)
    test_sentences = [
        "Oh fantastic, another flat tire on Monday morning.",
        "The pilot had a severe fear of heights.",
        "Her smile was a ray of sunshine in the dark.",
        "The conference will take place tomorrow at ten o'clock."
    ]
    results = predictor.predict_batch(test_sentences)
    for r in results:
        print(f"Text:       {r['text']}")
        print(f"Prediction: {r['predicted_class']} (Confidence: {r['confidence']:.4f})")
        print(f"Probs:      {r['probabilities']}\n")

## 21. Conclusions & Research Insights
- **Contextual Understanding**: Pretrained bidirectional transformer representations allow BERT to capture nuanced lexical juxtapositions without manual feature engineering.
- **Sarcasm vs Irony**: Contextual attention helps differentiate between sharp mocking intent (Sarcasm) and situational discrepancy (Irony), though subtle short texts remain challenging.
- **Computational Trade-Off**: BERT achieves competitive classification with ~109.5M parameters, requiring higher memory and inference latency compared to lightweight classical linear baselines.
- **Progression to Phase 9**: Results are packaged for final cross-model synthesis, error taxonomy analysis, and final production model selection.